In [1]:
import glob
import pandas as pd
import os

# Find all Excel files in the current directory
excel_files = glob.glob('*.xlsx') + glob.glob('*.xls')

if not excel_files:
    print("No Excel files found in the current folder.")
else:
    print(f"Found {len(excel_files)} Excel file(s): {excel_files}")

    # List to store dataframes
    df_list = []

    for file in excel_files:
        try:
            # Read each Excel file
            df = pd.read_excel(file)
            # Optional: Add a column to keep track of the source file
            df['source_file'] = file
            df_list.append(df)
            print(f"Successfully loaded: {file}")
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if df_list:
        # Concatenate all dataframes
        compiled_df = pd.concat(df_list, ignore_index=True)

        # # Save the compiled dataframe to a new Excel file
        # output_filename = 'compiled_excel_output.xlsx'
        # compiled_df.to_excel(output_filename, index=False)
        # print(f"\nAll files compiled successfully! Saved as: {output_filename}")

        # Display the first few rows of the compiled data
        display(compiled_df.head())
    else:
        print("No data could be compiled.")

Found 3 Excel file(s): ['Investor_2026-11-12_22-50-00.xlsx', 'Investor_2026-11-11_22-55-05.xlsx', 'Investor_2026-11-14_22-30-08.xlsx']
Successfully loaded: Investor_2026-11-12_22-50-00.xlsx
Successfully loaded: Investor_2026-11-11_22-55-05.xlsx
Successfully loaded: Investor_2026-11-14_22-30-08.xlsx


,Judul,Detail,Tanggal,Link,Kategori,Source,Crawled Date,source_file
0,BERITA POPULER: Grup Sinar Mas Jadi Pengendali...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",2026-11-13 05:33:00,https://investor.id/market/461794/berita-popul...,Market,Investor,2026-11-12 22:50:00,Investor_2026-11-12_22-50-00.xlsx
1,"MSCI Depak Sejumlah Saham RI, OJK Siapkan Lang...","JAKARTA, Investor.id– Otoritas Jasa Keuangan (...",2026-11-13 05:30:00,https://investor.id/market/461785/msci-depak-s...,Market,Investor,2026-11-12 22:50:00,Investor_2026-11-12_22-50-00.xlsx
2,"IHSG Pakai 522 Saham, Ini Top 10 dengan Bobot ...","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",2026-11-13 05:23:00,https://investor.id/market/461793/ihsg-pakai-5...,Market,Investor,2026-11-12 22:50:00,Investor_2026-11-12_22-50-00.xlsx
3,"Nvidia Anjlok 4%, Wall Street Kompak Berbalik ...","NEW YORK, investor.id– Wall Street kompak mele...",2026-11-13 04:30:00,https://investor.id/market/461792/nvidia-anjlo...,Market,Investor,2026-11-12 22:50:00,Investor_2026-11-12_22-50-00.xlsx
4,Harga Emas Terjun Bebas ke Level Terendah dala...,"NEW YORK, investor.id– Harga emas dunia terjun...",2026-11-13 04:19:00,https://investor.id/market/461791/harga-emas-t...,Market,Investor,2026-11-12 22:50:00,Investor_2026-11-12_22-50-00.xlsx


In [2]:
import pandas as pd

# Ensure the 'Tanggal' column is converted to datetime format
compiled_df['Tanggal_parsed'] = pd.to_datetime(compiled_df['Tanggal'], errors='coerce')

# Extract the day of the week (Monday=0, Tuesday=1, Wednesday=2, Thursday=3, Friday=4, Saturday=5, Sunday=6)
compiled_df['day_of_week'] = compiled_df['Tanggal_parsed'].dt.dayofweek

# Group 1: Thursday (3) to Sunday (6)
category_thu_sun = compiled_df[compiled_df['day_of_week'].isin([3, 4, 5, 6])].copy()

# Group 2: Monday (0) to Wednesday (2)
# category_mon_wed = compiled_df[compiled_df['day_of_week'].isin([0, 1, 2])].copy()

# Drop helper columns from output
for df_cat in [category_thu_sun]:
    df_cat.drop(columns=['Tanggal_parsed', 'day_of_week', 'source_file'], inplace=True, errors='ignore')

# # Save the filtered categories to separate Excel files
# category_thu_sun.to_excel('news_thursday_to_sunday.xlsx', index=False)
# category_mon_wed.to_excel('news_monday_to_wednesday.xlsx', index=False)

# print(f"Category 1 (Thursday - Sunday): {len(category_thu_sun)} rows saved to 'news_thursday_to_sunday.xlsx'")
# print(f"Category 2 (Monday - Wednesday): {len(category_mon_wed)} rows saved to 'news_monday_to_wednesday.xlsx'")

# Display previews
print("\n--- Preview: Thursday to Sunday ---")
display(category_thu_sun.head(2))

# print("\n--- Preview: Monday to Wednesday ---")
# display(category_mon_wed.head(2))


--- Preview: Thursday to Sunday ---


,Judul,Detail,Tanggal,Link,Kategori,Source,Crawled Date
0,BERITA POPULER: Grup Sinar Mas Jadi Pengendali...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",2026-11-13 05:33:00,https://investor.id/market/461794/berita-popul...,Market,Investor,2026-11-12 22:50:00
1,"MSCI Depak Sejumlah Saham RI, OJK Siapkan Lang...","JAKARTA, Investor.id– Otoritas Jasa Keuangan (...",2026-11-13 05:30:00,https://investor.id/market/461785/msci-depak-s...,Market,Investor,2026-11-12 22:50:00


In [3]:
from datetime import datetime

# 1. Get current system date and time
now = datetime.now()

# 2. Format it safely for a filename (Year-Month-Day_Hour-Minute-Second)
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

In [4]:
import os

# Define the name of the new folder
target_folder = 'categorized_news'

# Create the folder if it does not exist
os.makedirs(target_folder, exist_ok=True)

# Define the target file paths
thu_sun_path = os.path.join(target_folder, f'news_thursday_to_sunday_{timestamp}.xlsx')
# mon_wed_path = os.path.join(target_folder, f'news_monday_to_wednesday_{timestamp}.xlsx')

# Save the filtered dataframes into the new folder
category_thu_sun.to_excel(thu_sun_path, index=False)
# category_mon_wed.to_excel(mon_wed_path, index=False)

print(f"Created directory: '{target_folder}'")
print(f"Successfully saved Category 1 to: '{thu_sun_path}'")
# print(f"Successfully saved Category 2 to: '{mon_wed_path}'")

Created directory: 'categorized_news'
Successfully saved Category 1 to: 'categorized_news/news_thursday_to_sunday_2026-11-15_21-37-17.xlsx'
